Trying with pyusb now...

In [1]:
import usb.core
import usb.util
import time

# Constants for Gilson
VID = 0x1FD0
PID = 0x0404

# Step 1: Find the device
dev = usb.core.find(idVendor=VID, idProduct=PID)

if dev is None:
    raise ValueError("Gilson 4020 USB device not found.")

print(f"Found device: VID={hex(dev.idVendor)}, PID={hex(dev.idProduct)}")

# Step 2: Get basic info
try:
    print("Manufacturer:", usb.util.get_string(dev, dev.iManufacturer))
    print("Product     :", usb.util.get_string(dev, dev.iProduct))
    print("Serial Num  :", usb.util.get_string(dev, dev.iSerialNumber))
except usb.core.USBError as e:
    print("Descriptor read error:", e)

# Step 3: Set configuration
dev.set_configuration()
cfg = dev.get_active_configuration()

print("\n--- Interfaces & Endpoints ---")
for intf in cfg:
    print(f"Interface {intf.bInterfaceNumber}")
    for ep in intf:
        print(f"  Endpoint Address: {hex(ep.bEndpointAddress)}")
        print(f"    Attributes    : {hex(ep.bmAttributes)}")
        print(f"    MaxPacketSize : {ep.wMaxPacketSize}")
        print(f"    Direction     : {'IN' if usb.util.endpoint_direction(ep.bEndpointAddress) == usb.util.ENDPOINT_IN else 'OUT'}")

# Step 4: Attempt a control transfer (e.g., send '%' command)
print("\n--- Sending Raw Command ---")
try:
    response = dev.ctrl_transfer(
        bmRequestType=0x21,  # Host to device | Class | Interface
        bRequest=0x09,       # SET_REPORT (common for HID-like USB class)
        wValue=0x0200,       # Report Type and ID (guessed)
        wIndex=0,            # Interface (may need to change)
        data_or_wLength=[0x25]  # ASCII '%' command
    )
    print("Transfer succeeded.")
except usb.core.USBError as e:
    print("Control transfer failed:", e)


Found device: VID=0x1fd0, PID=0x404


ValueError: The device has no langid (permission issue, no string descriptors supported or device error)